In [ ]:
"""
A script that merges multiple xml files into a single pdf for better visual comparisons
"""

import music21 as m21
from collections import defaultdict
from pathlib import Path
from pypdf import PdfWriter

parent_dir = Path("playground")
output_dir = parent_dir / "merged_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

xml_files = sorted(list(parent_dir.rglob("*.xml")))

# group files by their exact filename
file_groups = defaultdict(list)
for file_path in xml_files:
    if output_dir in file_path.parents:
        continue
    file_groups[file_path.name].append(file_path)

print(f"Found {len(file_groups)} unique filename group(s). Starting merge process...\n")

for filename, paths in file_groups.items():
    print(f"Processing group: {filename} ({len(paths)} file(s))")
    
    merger = PdfWriter()
    temp_pdfs = []
    

    for i, path in enumerate(paths):

        score = m21.converter.parse(str(path))
        
        # Use only the parent directory as the rendered title
        dir_title = path.parent.name


        for element in list(score.recurse()):
            if isinstance(element, (m21.expressions.TextExpression, m21.text.TextBox)):
                active_site = element.activeSite
                if active_site is not None:
                    active_site.remove(element)

        score.metadata = m21.metadata.Metadata()
        score.metadata.title = dir_title

        title_exp = m21.expressions.TextExpression(dir_title)
        title_exp.placement = "above"
        score.insert(0, title_exp)

        # 3. Export to a temporary PDF
        temp_pdf = output_dir / f"temp_{i}_{path.stem}.pdf"
        temp_pdfs.append(temp_pdf)
        score.write('musicxml.pdf', fp=temp_pdf)
        
        # 4. Append to merger
        merger.append(str(temp_pdf))
        
    # output final combined PDF
    output_pdf_path = output_dir / f"merged_{Path(filename).stem}.pdf"
    with open(output_pdf_path, "wb") as output_file:
        merger.write(output_file)
    print(f"saved merged PDF to: {output_pdf_path}\n")
    merger.close()

    for temp_pdf in temp_pdfs:
        if temp_pdf.exists():
            temp_pdf.unlink()

    for temp_file in parent_dir.rglob("temp_*.xml"):
        if temp_file.exists():
                temp_file.unlink()
